# 07 — Inventory-constrained paired pilot
Prepares pilot-003 against the exact frozen pilot-002 sample; no resampling.

## 1. Paths and configuration

In [ ]:
from pathlib import Path

from rubric_order.api import find_project_root, load_experiment_config

PROJECT_ROOT = find_project_root(Path.cwd())
CONFIG_PATH = PROJECT_ROOT / "configs" / "pilot_v3.yaml"
DOWNSTREAM_CONFIG_PATH = PROJECT_ROOT / "configs" / "pilot_002_downstream_audit.yaml"
RUN_ID = "pilot-003"
EXECUTE_MODEL_CALLS = False
MAX_CALLS = 0
config = load_experiment_config(CONFIG_PATH)
downstream_config = load_experiment_config(DOWNSTREAM_CONFIG_PATH)
target = PROJECT_ROOT / config["output_root"] / RUN_ID

## 2. Provenance

In [ ]:
from rubric_order.v3_design import validate_paired_source

paired_source = validate_paired_source(config)
assert paired_source["status"] == "paired_source_verified"

## 3. Approved-judge gate

In [ ]:
from rubric_order.downstream import verify_approved_judge

judge_gate = verify_approved_judge(config)
assert judge_gate["status"] == "JUDGE_APPROVED"

## 4. Data and artifact integrity

In [ ]:
assert paired_source["checks"]["hashes_match"]
assert paired_source["checks"]["orders_match"]
assert paired_source["checks"]["heldout_match"]

## 5. Exact estimator

In [ ]:
from rubric_order.v3_design import estimate_pilot_v3

estimate = estimate_pilot_v3(config)
assert (estimate["updater_calls"], estimate["judge_calls"], estimate["total_inference_calls"]) == (
    132,
    500,
    632,
)

## 6. Phase-aware dry-run

In [ ]:
from rubric_order.v3_design import plan_pilot_v3

dry_run = plan_pilot_v3(config, RUN_ID)
assert dry_run["updater"]["planned_physical_updater_calls"] == 132
assert dry_run["updater"]["invalid_requests"] == 0
assert dry_run["ollama_contacted"] is False

## 7. Explicit execution — disabled
The paired pilot requires its complete 132-updater plus 500-judge budget and an absent explicit target.

In [ ]:
from rubric_order.v3_design import run_pilot_v3_judge, run_pilot_v3_updater

if EXECUTE_MODEL_CALLS:
    assert MAX_CALLS >= 632
    assert dry_run["ready"] and judge_gate["status"] == "JUDGE_APPROVED"
    assert RUN_ID == "pilot-003" and not target.exists(), f"target conflict: {target}"
    updater_execution = run_pilot_v3_updater(
        config, run_id=RUN_ID, execute_model_calls=True, max_calls=132
    )
    judge_execution = run_pilot_v3_judge(
        config, run_id=RUN_ID, execute_model_calls=True, max_calls=500
    )
    execution = {"updater": updater_execution, "judge": judge_execution}
else:
    assert MAX_CALLS == 0
    execution = {"executed": False, "actual_calls": 0, "ollama_contacted": False}

## 8. Resume and status

In [ ]:
manifest_path = target / "manifest.json"
resume_status = {"target_exists": target.exists(), "manifest_exists": manifest_path.exists()}

## 9. Paired analysis

In [ ]:
from rubric_order.v3_design import analyze_paired_pilot_v3

free_predictions = (
    PROJECT_ROOT / "runs" / "pilot-002-downstream-audit" / "artifacts" / "predictions.jsonl"
)
constrained_predictions = target / "artifacts" / "predictions.jsonl"
analysis = (
    analyze_paired_pilot_v3(
        free_predictions, constrained_predictions, target / "metrics" / "paired"
    )
    if free_predictions.exists() and constrained_predictions.exists()
    else {"status": "awaiting_saved_prediction_artifacts", "paired": True}
)

## 10. Readiness

In [ ]:
from rubric_order.v3_design import pilot_v3_readiness

downstream_target = PROJECT_ROOT / "runs" / "pilot-002-downstream-audit"
readiness = pilot_v3_readiness(config, downstream_target, downstream_config=downstream_config)

## 11. Unload model

In [ ]:
unload_status = {
    "handled_by_execution_stage": bool(EXECUTE_MODEL_CALLS),
    "model_loaded_by_notebook": False,
}